In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
from IPython.display import display
from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.LinearRegression import signature_helpers as sh
from metab_processing.LinearRegression.least_squares import (
    fit_gene_betas, subsample_gene_betas, rank_coefficients)

# Merged colon dataset (built once by colon_test.ipynb's REBUILD_FULL step).
FULL_DATA_PATH = f'{DATA_DIR}/Alexi_UC_Full_Data/processed_adata.h5ad'
ANNOT_COL, CELL_TYPE = '25_06_11_ICI_5K_Coarse_annotations', 'IEC'
COND_COL = '240927_Condition'              # UC / HC / ICI
PATIENT_COL = '25_06_11_Patient_ID_HS'
SAMPLE_COL = 'slice'                        # per-sample label added by merge_x_adatas
METHOD, PENALTY, STANDARDIZE = 'l1', 0.005, True
N_SUB = 50
LOWER, UPPER = 15, 85
DBVER = '2024.1.Hs'

SINGLE_GENE = 'CD3E'                         # single-gene reference, to contrast with signatures

SIGNATURES = [
    'GOBP_INTESTINAL_EPITHELIAL_CELL_DIFFERENTIATION',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION',
    'HALLMARK_GLYCOLYSIS',
    'HALLMARK_HYPOXIA',
    'HALLMARK_FATTY_ACID_METABOLISM',
    'HALLMARK_MTORC1_SIGNALING',
    'HALLMARK_APICAL_JUNCTION',
    'HALLMARK_APICAL_SURFACE',
    'HALLMARK_BILE_ACID_METABOLISM',
    'HALLMARK_XENOBIOTIC_METABOLISM',
    'GOBP_BICARBONATE_TRANSPORT',
    'GOBP_SODIUM_ION_TRANSPORT',
    'GOBP_DEFENSE_RESPONSE_TO_BACTERIUM',
    'GOBP_ANTIMICROBIAL_HUMORAL_RESPONSE',
    'HALLMARK_PANCREAS_BETA_CELLS',
    'HALLMARK_SPERMATOGENESIS',
]

In [ ]:
adata = sc.read_h5ad(FULL_DATA_PATH)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')
print('\nconditions:', adata.obs[COND_COL].value_counts().to_dict())

In [ ]:
library = sh.load_msigdb(DBVER)
print('loaded', len(library), 'gene sets')

## Target signatures: membership, missing, present, factor-removed, remaining

In [ ]:
sets = sh.resolve_sets(SIGNATURES, library, var, label='colon target signatures')
sh.report_factor_removal(adata, sets)

## Batch-covariate drivers

Same per-signature prediction as `signature_analysis`, but run across **sample subsets**
(all / UC / HC / ICI) and **batch covariates** (none / condition / patient / slice), each an
unpenalized one-hot (FWL) so per-batch level shifts don't leak into the factor betas.
`run_colon_gene` does the same for a single raw gene (the reference at the top).

In [ ]:
def run_colon_gene(target=SINGLE_GENE, n_sub=N_SUB):
    """Single-gene reference across the same subset x covariate grid as the signatures."""
    for oh in [None, COND_COL, PATIENT_COL, SAMPLE_COL]:
        _run_gene_one(adata, target, oh, 'All colon samples', n_sub)
    for cond in ['UC', 'HC', 'ICI']:
        sub_ad = adata[adata.obs[COND_COL] == cond].copy()
        for oh in [None, PATIENT_COL, SAMPLE_COL]:
            _run_gene_one(sub_ad, target, oh, f'Only {cond} samples', n_sub)


def _run_gene_one(ad, target, onehot_col, samples_name, n_sub):
    print(f'\n===== {target} | {samples_name} | covariate={onehot_col} =====')
    try:
        b = fit_gene_betas(ad, genes=[target], annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                           metabolites=[], custom='all', method=METHOD, penalty=PENALTY,
                           standardize=STANDARDIZE, onehot_col=onehot_col)
        sub, sub_r = subsample_gene_betas(ad, target, factors=None, n_subsamples=n_sub,
                                          annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                                          metabolites=[], custom='all', method=METHOD, penalty=PENALTY,
                                          standardize=STANDARDIZE, onehot_col=onehot_col, return_r=True,
                                          just_skip_samples_without_all_labels=True)
    except Exception as e:
        print(f'  run failed ({type(e).__name__}: {e})'); return
    res = sh.add_subsample_percentiles(rank_coefficients(b), sub, sub_r, LOWER, UPPER)
    display(res.head(40))
    sh.filter_and_plot_ratio(res).set_title(f'{target} | {samples_name} | covariate={onehot_col} | |avg_beta|/INTERVAL')
    sh.filter_and_plot_variance_ratio(res).set_title(f'{target} | {samples_name} | covariate={onehot_col} | |avg_beta|/VARIANCE')


def run_colon_signature(name, n_sub=N_SUB):
    """Predict signature `name` across the subset x batch-covariate grid."""
    for oh in [None, COND_COL, PATIENT_COL, SAMPLE_COL]:
        sh.run_one(adata, name, sets, annot_col=ANNOT_COL, annot_value=CELL_TYPE, onehot_col=oh,
                   samples_name='All colon samples', method=METHOD, penalty=PENALTY,
                   standardize=STANDARDIZE, n_sub=n_sub, custom='all', metabolites=[])
    for cond in ['UC', 'HC', 'ICI']:
        sub_ad = adata[adata.obs[COND_COL] == cond].copy()
        for oh in [None, PATIENT_COL, SAMPLE_COL]:
            sh.run_one(sub_ad, name, sets, annot_col=ANNOT_COL, annot_value=CELL_TYPE, onehot_col=oh,
                       samples_name=f'Only {cond} samples', method=METHOD, penalty=PENALTY,
                       standardize=STANDARDIZE, n_sub=n_sub, custom='all', metabolites=[])

## Single-gene reference: CD3E (shows the difference vs a signature target)

In [ ]:
run_colon_gene(SINGLE_GENE)

## Per-signature predictions (subset x batch-covariate grid)

In [ ]:
run_colon_signature('GOBP_INTESTINAL_EPITHELIAL_CELL_DIFFERENTIATION')

In [ ]:
run_colon_signature('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

In [ ]:
run_colon_signature('HALLMARK_INFLAMMATORY_RESPONSE')

In [ ]:
run_colon_signature('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
run_colon_signature('HALLMARK_GLYCOLYSIS')

In [ ]:
run_colon_signature('HALLMARK_HYPOXIA')

In [ ]:
run_colon_signature('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
run_colon_signature('HALLMARK_MTORC1_SIGNALING')

In [ ]:
run_colon_signature('HALLMARK_APICAL_JUNCTION')

In [ ]:
run_colon_signature('HALLMARK_APICAL_SURFACE')

In [ ]:
run_colon_signature('HALLMARK_BILE_ACID_METABOLISM')

In [ ]:
run_colon_signature('HALLMARK_XENOBIOTIC_METABOLISM')

In [ ]:
run_colon_signature('GOBP_BICARBONATE_TRANSPORT')

In [ ]:
run_colon_signature('GOBP_SODIUM_ION_TRANSPORT')

In [ ]:
run_colon_signature('GOBP_DEFENSE_RESPONSE_TO_BACTERIUM')

In [ ]:
run_colon_signature('GOBP_ANTIMICROBIAL_HUMORAL_RESPONSE')

In [ ]:
run_colon_signature('HALLMARK_PANCREAS_BETA_CELLS')

In [ ]:
run_colon_signature('HALLMARK_SPERMATOGENESIS')